# RainVision: Models & Rainfall Prediction Pipeline
## End-to-End Model Training, Evaluation, and Real-Time Inference

This notebook contains **only the machine learning models and predictive pipeline** of the RainVision platform:
- Quantitative rainfall depth regressors (Linear Regression, Ridge, Decision Tree, Random Forest, Gradient Boosting, Two-Stage Hurdle Model)
- Multi-class storm severity classifiers (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting)
- Comprehensive test set benchmarking & evaluation metrics
- Multi-model real-time inference engine with physical boundary checks, thermodynamic dew point physics, and automated public safety advisories

In [ ]:
# =====================================================================
# 1. Environment Setup & Data Loading
# =====================================================================
import os
import sys
import json
import time
import datetime
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
for folder in ["data", "models", "outputs/predictions", "outputs/figures", "outputs/reports"]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import joblib

import matplotlib
if not ("google.colab" in sys.modules or "ipykernel" in sys.modules):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 5)

from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

# Load master dataset
data_candidates = [
    PROJECT_ROOT / "data" / "RainVision_Historical_1985_to_Today.xlsx",
    PROJECT_ROOT / "RainVision_Historical_1985_to_Today.xlsx",
    Path("data/RainVision_Historical_1985_to_Today.xlsx"),
    Path("../data/RainVision_Historical_1985_to_Today.xlsx")
]

data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Master Excel dataset not found. Please place RainVision_Historical_1985_to_Today.xlsx in data/.")

df_raw = pd.read_excel(data_path, sheet_name="Rainfall_Data_1985_to_Today")
print(f"Dataset Loaded: {df_raw.shape[0]:,} records from {df_raw['date'].min()} to {df_raw['date'].max()}")

## 2. Feature Extraction & Chronological Splitting

We extract the **17 canonical meteorological predictors** (antecedent precipitation lags, multi-day rolling moisture, barometric tendency, solar cyclical angles) and partition chronologically (80% Train, 20% Held-out Test) to avoid temporal lookahead data leakage.

In [ ]:
# =====================================================================
# 2. Feature Extraction & Chronological Splitting
# =====================================================================
def categorize_rainfall(y: float) -> str:
    if y <= 0.05:
        return "Dry / Trace"
    elif y <= 2.5:
        return "Light"
    elif y <= 15.0:
        return "Moderate"
    elif y <= 50.0:
        return "Heavy"
    else:
        return "Torrential"

def prepare_features(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.drop_duplicates(subset=["date"]).sort_values("date").reset_index(drop=True)
    doy = df["date"].dt.dayofyear
    
    # Cyclical coordinates
    df["sin_day_of_year"] = np.sin(2 * np.pi * doy / 365.25)
    df["cos_day_of_year"] = np.cos(2 * np.pi * doy / 365.25)
    
    # Antecedent lags
    df["rain_lag_1"] = df["rainfall"].shift(1).fillna(0.0)
    df["rain_lag_2"] = df["rainfall"].shift(2).fillna(0.0)
    df["rain_lag_3"] = df["rainfall"].shift(3).fillna(0.0)
    
    # Rolling window moisture
    df["rain_roll_mean_7"] = df["rainfall"].shift(1).rolling(7, min_periods=1).mean().fillna(0.0)
    df["rain_roll_sum_7"] = df["rainfall"].shift(1).rolling(7, min_periods=1).sum().fillna(0.0)
    df["rain_roll_max_7"] = df["rainfall"].shift(1).rolling(7, min_periods=1).max().fillna(0.0)
    df["rain_roll_mean_14"] = df["rainfall"].shift(1).rolling(14, min_periods=1).mean().fillna(0.0)
    
    # Atmospheric trends
    df["temp_lag_1"] = df["temperature"].shift(1).bfill()
    df["humidity_lag_1"] = df["humidity"].shift(1).bfill()
    df["pressure_diff_1"] = (df["pressure"] - df["pressure"].shift(1)).fillna(0.0)
    
    df["is_monsoon"] = df["date"].dt.month.isin([6, 7, 8, 9]).astype(int)
    df["rainfall_category"] = df["rainfall"].apply(categorize_rainfall)
    return df

df_feat = prepare_features(df_raw)

FEATURE_COLUMNS = [
    "temperature", "humidity", "pressure", "wind_speed",
    "rain_lag_1", "rain_lag_2", "rain_lag_3",
    "rain_roll_mean_7", "rain_roll_sum_7", "rain_roll_max_7", "rain_roll_mean_14",
    "temp_lag_1", "humidity_lag_1", "pressure_diff_1",
    "sin_day_of_year", "cos_day_of_year", "is_monsoon"
]

CATEGORIES = ["Dry / Trace", "Light", "Moderate", "Heavy", "Torrential"]

# 80/20 chronological split
split_idx = int(len(df_feat) * 0.80)
train_df = df_feat.iloc[:split_idx].copy().reset_index(drop=True)
test_df = df_feat.iloc[split_idx:].copy().reset_index(drop=True)

X_train, y_train_reg, y_train_clf = train_df[FEATURE_COLUMNS], train_df["rainfall"], train_df["rainfall_category"]
X_test, y_test_reg, y_test_clf = test_df[FEATURE_COLUMNS], test_df["rainfall"], test_df["rainfall_category"]

print(f"Train Set : {len(train_df):,} days ({train_df['date'].min().date()} to {train_df['date'].max().date()})")
print(f"Test Set  : {len(test_df):,} days ({test_df['date'].min().date()} to {test_df['date'].max().date()})")

## 3. Quantitative Rainfall Depth Regression Suite

Training and benchmarking all regression models on unseen held-out test data:
- **Baseline Models:** Historical Mean, Persistence (Lag-1)
- **Linear Models:** Linear Regression (OLS), Ridge Regression (with `StandardScaler`)
- **Tree Models:** Decision Tree, Random Forest Regressor, Gradient Boosting Regressor
- **Two-Stage Hurdle Model:** Stage 1 Binary Classifier gating rain occurrence + Stage 2 Regressor for continuous depth

In [ ]:
# =====================================================================
# 3. Model Training: Quantitative Depth Regression
# =====================================================================
def evaluate_regression(y_true, y_pred, name):
    y_p = np.clip(np.asarray(y_pred, dtype=float), 0.0, None)
    y_t = np.asarray(y_true, dtype=float)
    mae = mean_absolute_error(y_t, y_p)
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    r2 = r2_score(y_t, y_p)
    wet = y_t > 0.05
    wet_mae = mean_absolute_error(y_t[wet], y_p[wet]) if np.sum(wet) > 0 else 0.0
    return {"Model": name, "MAE (mm)": round(mae, 3), "RMSE (mm)": round(rmse, 3), "R² Score": round(r2, 4), "Wet-Day MAE (mm)": round(wet_mae, 3)}

reg_suite = {
    "Baseline Mean": DummyRegressor(strategy="mean"),
    "Baseline Persistence (Lag-1)": "persistence",
    "Linear Regression (OLS)": LinearRegression(),
    "Ridge Regression": Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=10.0, random_state=42))]),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, min_samples_split=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=5, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
}

reg_results = []
trained_models = {}

print("Training Depth Regression Suite...")
for name, model in reg_suite.items():
    t0 = time.time()
    if name == "Baseline Persistence (Lag-1)":
        preds = X_test["rain_lag_1"].values
    else:
        model.fit(X_train, y_train_reg)
        preds = model.predict(X_test)
        trained_models[name] = model
    metrics = evaluate_regression(y_test_reg, preds, name)
    metrics["Latency (s)"] = round(time.time() - t0, 3)
    reg_results.append(metrics)

# Two-Stage Hurdle Model
print("Training Two-Stage Hurdle Model...")
t0 = time.time()
hurdle_gate = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
hurdle_gate.fit(X_train, (y_train_reg > 0.05).astype(int))

wet_mask = y_train_reg > 0.05
hurdle_amt = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=5, random_state=42, n_jobs=-1)
hurdle_amt.fit(X_train[wet_mask], y_train_reg[wet_mask])

p_gate = hurdle_gate.predict(X_test)
p_amt = np.clip(hurdle_amt.predict(X_test), 0.0, None)
hurdle_preds = np.where(p_gate == 1, p_amt, 0.0)

h_metrics = evaluate_regression(y_test_reg, hurdle_preds, "Two-Stage Hurdle Model")
h_metrics["Latency (s)"] = round(time.time() - t0, 3)
reg_results.append(h_metrics)
trained_models["Two-Stage Hurdle Model"] = (hurdle_gate, hurdle_amt)

df_reg_benchmark = pd.DataFrame(reg_results)
print("\n--- REGRESSION SUITE TEST RESULTS ---")
display(df_reg_benchmark.sort_values("R² Score", ascending=False))

## 4. Storm Severity Classification Suite

Training multi-class models to classify observations into severity tiers:
- *Dry / Trace* ($\le 0.05\text{ mm}$)
- *Light* ($0.05 < y \le 2.5\text{ mm}$)
- *Moderate* ($2.5 < y \le 15.0\text{ mm}$)
- *Heavy* ($15.0 < y \le 50.0\text{ mm}$)
- *Torrential* ($> 50.0\text{ mm}$)

In [ ]:
# =====================================================================
# 4. Model Training: Storm Severity Classification
# =====================================================================
def evaluate_clf(y_true, y_pred, name):
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Macro Precision": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "Macro Recall": round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "Macro F1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4)
    }

clf_suite = {
    "Baseline Majority": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))]),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_split=10, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=5, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
}

clf_results = []
print("Training Classification Suite...")
for name, model in clf_suite.items():
    model.fit(X_train, y_train_clf)
    preds = model.predict(X_test)
    trained_models[f"Classifier_{name}"] = model
    clf_results.append(evaluate_clf(y_test_clf, preds, name))

df_clf_benchmark = pd.DataFrame(clf_results)
print("\n--- CLASSIFICATION SUITE TEST RESULTS ---")
display(df_clf_benchmark.sort_values("Macro F1", ascending=False))

## 5. Actual vs Predicted Forecast Visualization

Visualizing model predictions against observed precipitation on unseen test data.

In [ ]:
# =====================================================================
# 5. Prediction Forecast Visualization
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Actual vs Predicted Scatter
rf_preds = np.clip(trained_models["Random Forest"].predict(X_test), 0.0, None)
axes[0].scatter(y_test_reg, rf_preds, alpha=0.35, color="#2980b9", edgecolors="none")
max_v = max(y_test_reg.max(), rf_preds.max())
axes[0].plot([0, max_v], [0, max_v], color="red", linestyle="--", label="Ideal 1:1 Line")
axes[0].set_title("Random Forest: Actual vs Predicted Rainfall (mm)", fontweight="bold")
axes[0].set_xlabel("Actual Rainfall (mm)")
axes[0].set_ylabel("Predicted Rainfall (mm)")
axes[0].legend()

# 60-Day Timeline Forecast Window
sample_test = test_df.iloc[-60:].copy().reset_index(drop=True)
sample_X = sample_test[FEATURE_COLUMNS]

rf_series = np.clip(trained_models["Random Forest"].predict(sample_X), 0.0, None)
lr_series = np.clip(trained_models["Linear Regression (OLS)"].predict(sample_X), 0.0, None)

axes[1].plot(sample_test["date"], sample_test["rainfall"], label="Actual Rainfall", color="black", linewidth=2.0)
axes[1].plot(sample_test["date"], rf_series, label="Random Forest", color="#27ae60", linestyle="--", linewidth=1.8)
axes[1].plot(sample_test["date"], lr_series, label="Linear Regression", color="#e74c3c", linestyle=":", linewidth=1.6)
axes[1].set_title("Recent 60-Day Forecast Timeline Comparison", fontweight="bold")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Rainfall (mm)")
axes[1].legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "figures" / "forecast_timeline_comparison.png", dpi=150)
plt.show()

## 6. Model Serialization

Saving verified production models into `models/` for downstream serving and dashboard deployment.

In [ ]:
# =====================================================================
# 6. Model Serialization
# =====================================================================
models_dir = PROJECT_ROOT / "models"
joblib.dump(trained_models["Random Forest"], models_dir / "random_forest_model.joblib")
joblib.dump(trained_models["Linear Regression (OLS)"], models_dir / "regression_model.joblib")
joblib.dump(trained_models["Gradient Boosting"], models_dir / "gradient_boosting_model.joblib")
joblib.dump(trained_models["Two-Stage Hurdle Model"], models_dir / "hurdle_model.joblib")
joblib.dump(trained_models["Classifier_Random Forest"], models_dir / "classifier_model.joblib")

metadata = {
    "timestamp": datetime.datetime.now().isoformat(),
    "feature_columns": FEATURE_COLUMNS,
    "categories": CATEGORIES,
    "models": {
        "random_forest": {"r2_score": float(df_reg_benchmark.loc[df_reg_benchmark["Model"]=="Random Forest", "R² Score"].values[0])},
        "linear_regression": {"r2_score": float(df_reg_benchmark.loc[df_reg_benchmark["Model"]=="Linear Regression (OLS)", "R² Score"].values[0])},
        "hurdle_model": {"r2_score": float(df_reg_benchmark.loc[df_reg_benchmark["Model"]=="Two-Stage Hurdle Model", "R² Score"].values[0])}
    }
}

with open(models_dir / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print(f"Models and metadata successfully saved to: {models_dir}")

## 7. Real-Time Meteorological Prediction & Weather Advisories

Executing live multi-model inference on new weather inputs:
- Validates physical domain limits ($0 \le T \le 55^\circ\text{C}$, $0\% \le RH \le 100\%$, $850 \le P \le 1100\text{ hPa}$, $W \ge 0$)
- Calculates thermodynamic Dew Point ($T_{\text{dew}} \approx T - \frac{100 - RH}{5}$) and Dew Point Depression ($\Delta T$)
- Predicts rainfall depth (mm) simultaneously from all models
- Issues automated civic public safety advisories based on predicted intensity

In [ ]:
# =====================================================================
# 7. Real-Time Meteorological Prediction Engine
# =====================================================================
BOUNDS = {
    "temperature": (0.0, 55.0, "°C"),
    "humidity": (0.0, 100.0, "%"),
    "pressure": (850.0, 1100.0, "hPa"),
    "wind_speed": (0.0, 180.0, "km/h")
}

def compute_dew_point(t: float, rh: float) -> float:
    return t - ((100.0 - rh) / 5.0)

def generate_civic_advisory(depth_mm: float, category: str) -> str:
    if depth_mm <= 0.05 or category == "Dry / Trace":
        return "Normal Weather: Clear / dry conditions. Routine outdoor operations may proceed."
    elif depth_mm <= 2.5 or category == "Light":
        return "Light Showers: Minor dampness. Carry an umbrella."
    elif depth_mm <= 15.0 or category == "Moderate":
        return "Moderate Rain Watch: Potential localized waterlogging. Drive with caution."
    elif depth_mm <= 50.0 or category == "Heavy":
        return "Heavy Rainfall Alert: Significant waterlogging hazard. Drainage systems under load."
    else:
        return "Torrential Storm Emergency: Flash flood risk. Avoid travel and evacuate low-lying spots."

def predict_rainvision(inputs: dict) -> dict:
    for k, (b_min, b_max, unit) in BOUNDS.items():
        if k not in inputs:
            raise ValueError(f"Missing meteorological input: '{k}'")
        val = float(inputs[k])
        if not (b_min <= val <= b_max):
            raise ValueError(f"Value for '{k}' ({val} {unit}) is outside domain bounds [{b_min}, {b_max} {unit}].")
            
    now = datetime.datetime.now()
    doy = now.timetuple().tm_yday
    
    vec = {
        "temperature": float(inputs["temperature"]),
        "humidity": float(inputs["humidity"]),
        "pressure": float(inputs["pressure"]),
        "wind_speed": float(inputs["wind_speed"]),
        "rain_lag_1": float(inputs.get("previous_rainfall", 0.0)),
        "rain_lag_2": float(inputs.get("rain_lag_2", 0.0)),
        "rain_lag_3": float(inputs.get("rain_lag_3", 0.0)),
        "rain_roll_mean_7": float(inputs.get("rain_roll_mean_7", 0.0)),
        "rain_roll_sum_7": float(inputs.get("rain_roll_sum_7", 0.0)),
        "rain_roll_max_7": float(inputs.get("rain_roll_max_7", 0.0)),
        "rain_roll_mean_14": float(inputs.get("rain_roll_mean_14", 0.0)),
        "temp_lag_1": float(inputs.get("temp_lag_1", inputs["temperature"])),
        "humidity_lag_1": float(inputs.get("humidity_lag_1", inputs["humidity"])),
        "pressure_diff_1": float(inputs.get("pressure_diff_1", 0.0)),
        "sin_day_of_year": np.sin(2 * np.pi * doy / 365.25),
        "cos_day_of_year": np.cos(2 * np.pi * doy / 365.25),
        "is_monsoon": 1 if now.month in [6, 7, 8, 9] else 0
    }
    
    in_df = pd.DataFrame([vec])[FEATURE_COLUMNS]
    
    # Run predictions across models
    rf_depth = float(np.clip(trained_models["Random Forest"].predict(in_df)[0], 0.0, None))
    lr_depth = float(np.clip(trained_models["Linear Regression (OLS)"].predict(in_df)[0], 0.0, None))
    gb_depth = float(np.clip(trained_models["Gradient Boosting"].predict(in_df)[0], 0.0, None))
    
    # Hurdle Model
    h_gate, h_amt = trained_models["Two-Stage Hurdle Model"]
    g_flag = h_gate.predict(in_df)[0]
    h_depth = float(np.clip(h_amt.predict(in_df)[0], 0.0, None)) if g_flag == 1 else 0.0
    
    # Classifier Category
    cat_pred = str(trained_models["Classifier_Random Forest"].predict(in_df)[0])
    
    # Thermodynamic physics
    dew_pt = compute_dew_point(vec["temperature"], vec["humidity"])
    dew_dep = vec["temperature"] - dew_pt
    advisory = generate_civic_advisory(rf_depth, cat_pred)
    
    return {
        "Random Forest (mm)": round(rf_depth, 2),
        "Linear Regression (mm)": round(lr_depth, 2),
        "Gradient Boosting (mm)": round(gb_depth, 2),
        "Two-Stage Hurdle (mm)": round(h_depth, 2),
        "Predicted Category": cat_pred,
        "Dew Point (°C)": round(dew_pt, 1),
        "Dew Point Depression (°C)": round(dew_dep, 1),
        "Civic Advisory": advisory
    }

# Predict on 4 Realistic Weather Scenarios
test_cases = {
    "Dry Pre-Monsoon Summer": {"temperature": 41.0, "humidity": 24.0, "pressure": 1012.0, "wind_speed": 12.0, "previous_rainfall": 0.0},
    "Pre-Monsoon Humidity Surge": {"temperature": 34.0, "humidity": 75.0, "pressure": 1003.0, "wind_speed": 18.0, "previous_rainfall": 1.5},
    "Active Monsoon Downpour": {"temperature": 27.5, "humidity": 94.0, "pressure": 997.0, "wind_speed": 28.0, "previous_rainfall": 35.0},
    "Severe Cyclonic Storm": {"temperature": 24.0, "humidity": 98.0, "pressure": 988.0, "wind_speed": 62.0, "previous_rainfall": 75.0, "pressure_diff_1": -8.5}
}

for title, weather in test_cases.items():
    print(f"\n==================== SCENARIO: {title.upper()} ====================")
    result = predict_rainvision(weather)
    for k, v in result.items():
        print(f"  {k:28s}: {v}")

## 8. Batch Prediction & CSV Export

Executing multi-model predictions on the unseen test set and exporting predictions to `outputs/predictions/sample_predictions.csv`.

In [ ]:
# =====================================================================
# 8. Batch Predictions & Export
# =====================================================================
batch_df = test_df.tail(100).copy().reset_index(drop=True)
batch_X = batch_df[FEATURE_COLUMNS]

batch_df["rf_predicted_rainfall"] = np.clip(trained_models["Random Forest"].predict(batch_X), 0.0, None).round(2)
batch_df["lr_predicted_rainfall"] = np.clip(trained_models["Linear Regression (OLS)"].predict(batch_X), 0.0, None).round(2)
batch_df["predicted_category"] = trained_models["Classifier_Random Forest"].predict(batch_X)

export_cols = ["date", "temperature", "humidity", "pressure", "wind_speed", "rainfall", "rf_predicted_rainfall", "lr_predicted_rainfall", "rainfall_category", "predicted_category"]
export_path = PROJECT_ROOT / "outputs" / "predictions" / "sample_predictions.csv"
batch_df[export_cols].to_csv(export_path, index=False)

print(f"Batch predictions exported to: {export_path}")
print(f"Exported {len(batch_df)} rows. Sample:")
display(batch_df[export_cols].head(5))